In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('Данные об аудитории.csv')
df.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


In [3]:
df['date'] = pd.to_datetime(df['date'])

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16814 entries, 0 to 16813
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          16814 non-null  datetime64[ns]
 1   user_id       16814 non-null  object        
 2   view_adverts  16814 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 394.2+ KB


# MAU

In [5]:
print(df['user_id'].nunique())

7639


# DAU mean

In [6]:
group = df.groupby('date')['user_id'].nunique().reset_index()
print(group['user_id'].mean())

560.4666666666667


In [7]:
group

,date,user_id
0,2023-11-01,623
1,2023-11-02,649
2,2023-11-03,573
3,2023-11-04,343
4,2023-11-05,350
5,2023-11-06,660
6,2023-11-07,629
7,2023-11-08,600
8,2023-11-09,661
9,2023-11-10,583


# RETENTION

In [8]:
cog = df[df['date'] == df['date'].min()]
cog

,date,user_id,view_adverts
7,2023-11-01,2f35ccf0-796b-11ed-96d9-2b24e5ad01fb,12
8,2023-11-01,ef6eb4d0-01bd-11ed-ab09-b1117ba72e2f,15
9,2023-11-01,17ce4000-77db-11ee-a3a0-edfd72e146b8,13
72,2023-11-01,d234f070-2821-11ed-9ce0-b7a9dd423e66,21
73,2023-11-01,77acbda0-786a-11ee-8810-032dc795ffb0,21
...,...,...,...
16707,2023-11-01,1fba7800-18db-11ea-a00b-cf3d2fa771e7,8
16752,2023-11-01,cf733600-e3ea-11ed-a5b9-b3c6a94e0cba,9
16772,2023-11-01,86576eb0-cedb-11ed-8a6f-f38a23eae62b,9
16773,2023-11-01,87b83df0-fb1f-11ed-a7d0-91e1f7cf1912,9


In [9]:
df_ret = df.merge(cog, on = 'user_id', how = 'inner')
df_retention = df_ret[df_ret['date_x'] == '2023-11-02']
print(df_retention['user_id'].nunique() / cog['user_id'].nunique() * 100)

26.64526484751204


# Конверсия

In [10]:
df[df['view_adverts'] == 0]

,date,user_id,view_adverts
374,2023-11-11,c5c3f8e0-805e-11ee-b828-4b47cdbdd931,0
375,2023-11-11,5f2afc00-78cf-11ee-b854-b99a46e8257e,0
376,2023-11-11,580b1600-5aeb-11ee-a31a-718c9cb74cd4,0
377,2023-11-11,fbaeb4c0-23a9-11ee-a5e0-63597bbc5fd5,0
378,2023-11-11,ccddf9f0-faef-11ec-a6ac-a31df73dcb6b,0
...,...,...,...
10427,2023-11-15,30a30da0-833d-11ee-9f8a-9b2d0e7116d9,0
10428,2023-11-15,aa687b50-57f9-11ec-8daa-e56655ea343f,0
10429,2023-11-15,abdda0a0-734c-11ec-89dd-4f6164320d3f,0
10430,2023-11-15,8b142ef0-0cf5-11ed-8af1-63871af601f1,0


In [11]:
user_group_1 = df[df['view_adverts'] != 0]
user_group_0 = df[df['view_adverts'] == 0]
print(user_group_1['user_id'].nunique() / df['user_id'].nunique() * 100)

46.31496269145176


# Среднее просмотренных объявлений на пользователя

In [12]:
df['view_adverts'].sum() / df['user_id'].nunique()

np.float64(2.8687000916350307)

# NPS

In [13]:
(1200 / 2000 * 100) - (500 / 2000 * 100)

35.0

# A/B test

In [14]:
df_ab = pd.read_csv('Данные АБ тестов.csv')
df_ab.head()

,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


In [15]:
df_ab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2835 entries, 0 to 2834
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   experiment_num    2835 non-null   int64 
 1   experiment_group  2835 non-null   object
 2   user_id           2835 non-null   int64 
 3   revenue           2835 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 88.7+ KB


In [16]:
ab_test1 = df_ab[df_ab['experiment_num'] == 1]
ab_test2 = df_ab[df_ab['experiment_num'] == 2]
ab_test3 = df_ab[df_ab['experiment_num'] == 3]

In [17]:
test1_control = ab_test1[ab_test1['experiment_group'] == 'control']
test1_test = ab_test1[ab_test1['experiment_group'] == 'test']

test2_control = ab_test2[ab_test2['experiment_group'] == 'control']
test2_test = ab_test2[ab_test2['experiment_group'] == 'test']

test3_control = ab_test3[ab_test3['experiment_group'] == 'control']
test3_test = ab_test3[ab_test3['experiment_group'] == 'test']

In [18]:
from scipy.stats import norm
from scipy.stats import t
from scipy.stats import stats

In [19]:
control1 = test1_control['revenue']
test1 = test1_test['revenue']

# Сначала посчитаю вручную, затем сравню с готовой функцией ttest_ind
control1_mean = control1.mean()
test1_mean = test1.mean()

control1_std = np.std(control1, ddof = 1)
test1_std = np.std(test1, ddof = 1)

control1_size = len(control1)
test1_size = len(test1)

dof1 = control1_size + test1_size - 2
pooled_var1 = ((control1_size - 1) * control1_std ** 2 + (test1_size - 1) * test1_std ** 2) / dof1
se1 = np.sqrt(pooled_var1 * (1 / control1_size + 1 / test1_size))

In [20]:
t_stat1 = (test1_mean - control1_mean) / se1
p_val1 = 2 * (1 - t.cdf(abs(t_stat1), dof1))

In [21]:
print(control1_mean, test1_mean)
print(dof1)
print(t_stat1)
print(p_val1)

722.4602150537635 665.7395833333334
943
-0.40062876605774095
0.6887842117790162


In [34]:
print('test 1:')
print(stats.ttest_ind(test1, control1, equal_var = True))

test 1:
TtestResult(statistic=np.float64(-0.4006287660577395), pvalue=np.float64(0.688784211779017), df=np.float64(943.0))


/var/folders/0j/sv6wvbnd3hnfh79b8rq4g0p40000gn/T/ipykernel_46414/573760683.py:2: DeprecationWarning: Please import `ttest_ind` from the `scipy.stats` namespace; the `scipy.stats.stats` namespace is deprecated and will be removed in SciPy 2.0.0.
  print(stats.ttest_ind(test1, control1, equal_var = True))


In [35]:
control2 = test2_control['revenue']
test2 = test2_test['revenue']
print('test 2:')
print(stats.ttest_ind(test2, control2, equal_var = True))

test 2:
TtestResult(statistic=np.float64(-3.303268516112734), pvalue=np.float64(0.0009915972237576197), df=np.float64(943.0))


/var/folders/0j/sv6wvbnd3hnfh79b8rq4g0p40000gn/T/ipykernel_46414/2563608108.py:4: DeprecationWarning: Please import `ttest_ind` from the `scipy.stats` namespace; the `scipy.stats.stats` namespace is deprecated and will be removed in SciPy 2.0.0.
  print(stats.ttest_ind(test2, control2, equal_var = True))


In [36]:
control3 = test3_control['revenue']
test3 = test3_test['revenue']
print('test 3:')
print(stats.ttest_ind(test3, control3, equal_var = True))

test 3:
TtestResult(statistic=np.float64(1.8703638838987622), pvalue=np.float64(0.061742900112187644), df=np.float64(943.0))


/var/folders/0j/sv6wvbnd3hnfh79b8rq4g0p40000gn/T/ipykernel_46414/2805375743.py:4: DeprecationWarning: Please import `ttest_ind` from the `scipy.stats` namespace; the `scipy.stats.stats` namespace is deprecated and will be removed in SciPy 2.0.0.
  print(stats.ttest_ind(test3, control3, equal_var = True))


# Листеры средний доход на пользователя

In [25]:
df_listers = pd.read_csv('Листеры.csv')
df_listers.head()

,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [26]:
df_listers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       158 non-null    int64 
 1   date          158 non-null    object
 2   cnt_adverts   158 non-null    int64 
 3   age           158 non-null    int64 
 4   cnt_contacts  158 non-null    int64 
 5   revenue       158 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 7.5+ KB


In [27]:
df_listers.describe().T

,count,mean,std,min,25%,50%,75%,max
user_id,158.0,115.101266,9.008976,100.0,107.00,115.0,123.0,130.0
cnt_adverts,158.0,2.886076,2.028380,0.0,1.00,3.0,5.0,6.0
age,158.0,27.424051,4.640303,20.0,22.75,28.0,31.0,36.0
cnt_contacts,158.0,94.031646,60.231128,0.0,42.00,92.5,148.5,200.0
revenue,158.0,30.702532,16.575060,0.0,17.00,31.0,45.0,60.0


In [28]:
df_listers['revenue'].sum() / df_listers['user_id'].nunique()

np.float64(156.48387096774192)

# Медиана

In [29]:
listers_group = df_listers.groupby('user_id')['age'].median().reset_index()
listers_group['age'].median()

28.0

# 18 задание, вывод по эксперименту

In [37]:
from statsmodels.stats.proportion import proportions_ztest

In [40]:
pays = [1099, 1003]
clients = [100001055, 100047501]

In [41]:
stat, p = proportions_ztest(pays, clients)
print(stat, p)

2.1045506019397746 0.03533044544854253


In [42]:
conver_a = 1003 / 100047501
conver_b = 1099 / 100001055
print(f'Прирост конверсии: {(conver_b - conver_a) / conver_a * 100}') 

Прирост конверсии: 9.622177084237158
